In [ ]:
user_command           = ""  # NL command, e.g. "update COGS for C1 to 25% in July 2024"
old_run_id             = ""  # UUID of the run to copy from (baseline scenario)
clarification_response = ""  # Leave empty on first run; fill in on re-run if clarification was requested


## Cell 1 — Load SKILL.md

In [ ]:
with open("/lakehouse/default/Files/skills/hive-optimizer-nl-intent/SKILL.md") as f:
    raw_skill = f.read()

# Strip YAML frontmatter
if raw_skill.startswith("---"):
    end = raw_skill.find("---", 3)
    raw_skill = raw_skill[end + 3:].lstrip("\n") if end != -1 else raw_skill

# Strip ## Examples section to reduce token usage
examples_idx = raw_skill.find("## Examples")
skill_md = raw_skill[:examples_idx].rstrip() if examples_idx != -1 else raw_skill

## Cell 2 — Call LiteLLM API with Schema Validation Retry

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ""

import litellm, json, jsonschema

MODEL = "anthropic/claude-haiku-4-5-20251001"
MAX_RETRIES = 3

with open("/lakehouse/default/Files/skills/hive-optimizer-nl-intent/intent_schema.json") as f:
    schema = json.load(f)

SYSTEM_MESSAGE = {
    "role": "system",
    "content": [
        {
            "type": "text",
            "text": skill_md,
            "cache_control": {"type": "ephemeral"},
        }
    ],
}


def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        text = text[text.index("\n") + 1:] if "\n" in text else text
        text = text.rsplit("```", 1)[0].strip()
    return text


# On re-run, append the user's clarification to the original command
if clarification_response:
    full_command = f"{user_command}\n\nUser clarification: {clarification_response}"
else:
    full_command = user_command

messages = [SYSTEM_MESSAGE, {"role": "user", "content": full_command}]
intent = None

for attempt in range(1, MAX_RETRIES + 1):
    print(f"Attempt {attempt}/{MAX_RETRIES}...")

    response = litellm.completion(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
    )

    raw = strip_fences(response.choices[0].message.content)
    messages.append({"role": "assistant", "content": raw})

    try:
        intent = json.loads(raw)
        jsonschema.validate(instance=intent, schema=schema)
        print(f"Schema validation passed on attempt {attempt}.")
        break
    except json.JSONDecodeError as e:
        error_msg = f"Your response was not valid JSON: {e}\nRespond with raw JSON only, no markdown fences."
    except jsonschema.ValidationError as e:
        path = " -> ".join(str(p) for p in e.absolute_path) or "(root)"
        error_msg = (
            f"Your response failed JSON schema validation.\n"
            f"Field path: {path}\n"
            f"Error: {e.message}\n"
            f"Fix that field and respond with the corrected raw JSON only."
        )

    if attempt < MAX_RETRIES:
        print(f"Error on attempt {attempt}: {error_msg}")
        messages.append({"role": "user", "content": error_msg})
    else:
        raise RuntimeError(f"Schema validation failed after {MAX_RETRIES} attempts. Last error: {error_msg}")

print(json.dumps(intent, indent=2))

## Cell 3 — Handle Clarifications

In [ ]:
if intent["period_resolution_required"] or intent["confidence"] < 0.80:
    print("=== Clarification needed ===")
    print(f"\n{intent['plain_english']}\n")
    print("Please answer the following, then re-run with clarification_response filled in:\n")
    for i, q in enumerate(intent["ambiguities"], 1):
        print(f"  {i}. {q}")
    mssparkutils.notebook.exit(json.dumps({
        "status": "clarification_needed",
        "plain_english": intent["plain_english"],
        "questions": intent["ambiguities"]
    }))


## Cell 4 — SQL Validation

In [ ]:
import re

ALLOWED_TABLES = {
    "input_cogs", "input_customer_capacities", "input_investor_capital",
    "input_performance_curves", "input_aggregated_raises", "input_aggregated_deployments",
    "input_fix_raises", "input_fix_deployments", "input_parameters",
    "input_investor_repayment_schedule", "input_net_new_capacities",
    "input_periods_config", "input_portfolios_config", "input_relative_deployments",
    "input_repeats_distribution", "input_time_periods"
}
ALLOWED_VERBS = {"UPDATE", "INSERT", "DELETE"}

def validate_sql(sql: str) -> None:
    upper = sql.strip().upper()
    verb = upper.split()[0]
    assert verb in ALLOWED_VERBS, f"Disallowed SQL verb '{verb}' — only UPDATE/INSERT/DELETE allowed"
    assert re.search(r"WHERE\s+RUNID\s*=\s*'\{RUN_ID\}'", upper), \
        f"SQL must contain exactly WHERE RunID = '{{run_id}}' — got: {sql[:120]}"
    assert any(t.upper() in upper for t in ALLOWED_TABLES), \
        f"SQL targets unknown or disallowed table: {sql[:80]}"

for stmt in intent["sql_mutations"]:
    validate_sql(stmt)
print(f"SQL validation passed ({len(intent['sql_mutations'])} statements).")

## Cell 5 — Get New RunID then CopyRun

In [ ]:
result = mssparkutils.notebook.run(
    "DE_NB_Get_New_Run_Id",
    timeout_seconds=60,
    arguments={
        "old_run_id": old_run_id_to_copy,
        "name": name,
        "description": description,
        "duration": duration,
        "user": user
    }
)
new_run_id = json.loads(result)["new_run_id"]
print(f"New RunID: {new_run_id}")

## Cell 5.5 — Preview: Rows That Will Be Updated

In [ ]:
import re

def _derive_preview_select(update_sql, preview_run_id):
    match = re.match(
        r"UPDATE\s+(\w+)\s+SET\s+.+?\s+(WHERE\s+.+)$",
        update_sql.strip(),
        re.IGNORECASE | re.DOTALL,
    )
    if not match:
        raise ValueError(f"Cannot parse UPDATE for preview: {update_sql!r}")
    table = match.group(1)
    where_clause = match.group(2)
    # Handle both {RUN_ID} and {run_id} casing variants found in the codebase
    resolved_where = (
        where_clause
        .replace("{RUN_ID}", preview_run_id)
        .replace("{run_id}", preview_run_id)
    )
    return f"SELECT * FROM {table} {resolved_where}"

print("=== Preview: rows that each UPDATE will touch (new_run_id data, pre-mutation) ===")
for i, stmt in enumerate(intent["sql_mutations"], 1):
    preview_sql = _derive_preview_select(stmt.strip(), new_run_id)
    print(f"\n--- Statement {i} ---")
    print(preview_sql)
    display(spark.sql(preview_sql))

## Cell 6 — Apply Scenario

In [ ]:
if intent["sql_mutations"]:
    sql_statements = ";\n".join(intent["sql_mutations"])
    mssparkutils.notebook.run(
        "DE_NB_Apply_Scenario",
        timeout_seconds=300,
        arguments={
            "run_id": new_run_id,
            "sql_statements": sql_statements
        }
    )
    print(f"Apply_Scenario complete: {len(intent['sql_mutations'])} statements executed.")
else:
    print("No SQL mutations \u2014 skipping Apply_Scenario.")


## Cell 7 — Run Model

In [ ]:
mssparkutils.notebook.run(
    "DE_NB_RunModel",
    timeout_seconds=intent["run_params"]["timeout_seconds"] + 120,
    arguments={
        "run_id": new_run_id,
        "run_params": json.dumps(intent["run_params"])
    }
)
print(f"RunModel complete. Results are in RunID: {new_run_id}")


## Cell 8 — Return Result

In [ ]:
result = {
    "new_run_id": new_run_id,
    "plain_english": intent["plain_english"],
    "sql_mutations_applied": len(intent["sql_mutations"])
}
print(json.dumps(result, indent=2))
mssparkutils.notebook.exit(json.dumps(result))
